# Setup 

In [17]:
import pandas as pd
import numpy as np


# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv("../data/mercari_sample.csv")

# ── Cleaning (mirrors 02_cleaning.ipynb) ──────────────────────────────────────
df["brand_name"] = df["brand_name"].fillna("No Brand")   # 43.2% null → sentinel
df = df.dropna(subset=["category_name"])                  # 0.47% null → drop
df = df[df["price"] > 0]                                  # free items → drop
df = df.drop_duplicates()                                  # defensive; 0 found in 02

# Normalise brand casing for consistent groupby keys
df["brand_name"] = df["brand_name"].str.lower().str.strip()


df["main_category"]    = df["category_name"].str.split("/").str[0]
df["sub_category"]     = df["category_name"].str.split("/").str[1]
df["brand_name"]       = df["brand_name"].str.lower().str.strip()
df["has_description"]  = df["item_description"].apply(
    lambda x: 0 if x == "No description yet" else 1)
df["name_length"]      = df["name"].str.len()
df["name_word_count"]  = df["name"].str.split().str.len()
df["desc_length"]      = df["item_description"].str.len()
df["is_branded"]       = np.where(df["brand_name"] == "no brand", 0, 1)
df["log_price"]        = np.log1p(df["price"])
df["category_depth"]   = df["category_name"].str.split("/").str.len()

condition_map = {1:"New", 2:"Like New", 3:"Good", 4:"Fair", 5:"Poor"}
df["condition_label"]  = df["item_condition_id"].map(condition_map)


cat_avg   = df.groupby("main_category")["price"].mean()
brand_avg = df.groupby("brand_name")["price"].mean()
df["cat_avg_price"]     = df["main_category"].map(cat_avg)
df["brand_avg_price"]   = df["brand_name"].map(brand_avg)
df["price_rank_in_cat"] = df.groupby("main_category")["price"]\
    .rank(ascending=False, method="dense").astype(int)
df["price_percentile"]  = df["price"].rank(pct=True).round(3)

print(f"Shape: {df.shape}")
print(f"Features ready for correlation analysis")

Shape: (49725, 22)
Features ready for correlation analysis


# Exploratory Data Analysis

In [18]:
# Average price by main category - sorted descending
print("=== Average price by main category ===")
df.groupby("main_category")["price"].mean().sort_values(ascending=False)

=== Average price by main category ===


main_category
Electronics               34.949905
Men                       34.895254
Vintage & Collectibles    29.276341
Women                     28.715048
Sports & Outdoors         25.737470
Home                      24.761604
Other                     20.543210
Kids                      20.205415
Beauty                    19.691000
Handmade                  18.260536
Name: price, dtype: float64

In [19]:
# Item count per main_category 
print("=== Item count per main category ===")
df.groupby("main_category").size().sort_values(ascending=False)

=== Item count per main category ===


main_category
Women                     22409
Beauty                     7089
Kids                       5725
Electronics                4222
Men                        3055
Home                       2219
Vintage & Collectibles     1585
Other                      1539
Handmade                   1044
Sports & Outdoors           838
dtype: int64

In [20]:
# Full agg table - avg, count, max, min per category
print("=== Price agg stats by main category ===")
df.groupby('main_category')['price'].agg(
    avg_price="mean",
    total_items="count",
    max_price="max",
    min_price="min"
)

=== Price agg stats by main category ===


,avg_price,total_items,max_price,min_price
main_category,,,,
Beauty,19.691000,7089,412.0,3.0
Electronics,34.949905,4222,859.0,3.0
Handmade,18.260536,1044,459.0,3.0
Home,24.761604,2219,360.0,3.0
Kids,20.205415,5725,525.0,3.0
Men,34.895254,3055,525.0,3.0
Other,20.543210,1539,650.0,3.0
Sports & Outdoors,25.737470,838,250.0,3.0
Vintage & Collectibles,29.276341,1585,1109.0,3.0


In [21]:
# Shipping rate per category - % seller pays
print("=== Shipping rate per main category ===")
df.groupby("main_category")["shipping"].mean().round(2)

=== Shipping rate per main category ===


main_category
Beauty                    0.60
Electronics               0.59
Handmade                  0.63
Home                      0.28
Kids                      0.42
Men                       0.35
Other                     0.54
Sports & Outdoors         0.46
Vintage & Collectibles    0.56
Women                     0.39
Name: shipping, dtype: float64

In [22]:
# Avg price by condition - does New cost more than poor?
print("=== Average price by item condition ===")
df.groupby("condition_label")["price"].mean().sort_values(ascending=False)

=== Average price by item condition ===


condition_label
Poor        34.297872
Like New    27.741928
Good        26.637955
New         26.272196
Fair        22.158965
Name: price, dtype: float64

In [23]:
# Average price by brand - top 10 excludeing No Brand
print("\n=== Average price by brand (top 10) ===")
df[df["brand_name"] != "No Brand"].groupby("brand_name")["price"].mean().sort_values(ascending=False).head(10)


=== Average price by brand (top 10) ===


brand_name
celine               539.500000
generic              486.000000
canada goose         415.000000
valentino            412.500000
alexander mcqueen    398.666667
chloé                368.500000
saint laurent        302.800000
vitamix              300.000000
bottega veneta       300.000000
samsung galaxy       276.000000
Name: price, dtype: float64

In [24]:
# Avg price per category x shipping - does free shipping correlate with higher price?
pd.pivot_table(df, values="price", index="main_category", columns="shipping",
               aggfunc="mean", fill_value=0).round(2)

shipping,0,1
main_category,,
Beauty,23.47,17.15
Electronics,48.05,25.76
Handmade,25.94,13.74
Home,26.72,19.77
Kids,22.60,16.84
Men,36.38,32.12
Other,24.89,16.80
Sports & Outdoors,27.21,23.99
Vintage & Collectibles,37.64,22.81


In [25]:
# Description premium - has_description = 1 vs 0 by category
print("\n=== Description premium by category ===")
df.groupby(["main_category", "has_description"])["price"].mean().unstack()


=== Description premium by category ===


has_description,0,1
main_category,,
Beauty,20.901515,19.644176
Electronics,17.550562,36.124526
Handmade,13.000000,18.421520
Home,22.309278,24.996543
Kids,18.803763,20.302821
Men,30.952381,35.155269
Other,22.819277,20.413462
Sports & Outdoors,21.558140,25.963522
Vintage & Collectibles,20.956044,29.783133


In [26]:
# IQR outlier analysis - bounds, count, % of dataset
print("\n=== Price outlier analysis ===")
Q1 = np.percentile(df["price"], 25)
Q3 = np.percentile(df["price"], 75)         
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR        
outliers = df[(df["price"] < lower_bound) | (df["price"] > upper_bound)]
print(f"Lower bound: ${lower_bound:.2f}")
print(f"Upper bound: ${upper_bound:.2f}")
print(f"Outlier count: {len(outliers)}")

# Flag Outliers with np.where
df["is_price_outlier"] = np.where(
    (df["price"] < lower_bound) | (df["price"] > upper_bound), 1, 0
)
print(f"Outliers as % of dataset: {df['is_price_outlier'].mean():.2%}")
print(df['is_price_outlier'].value_counts())


=== Price outlier analysis ===
Lower bound: $-18.50
Upper bound: $57.50
Outlier count: 3972
Outliers as % of dataset: 7.99%
is_price_outlier
0    45753
1     3972
Name: count, dtype: int64


In [27]:
# Correlation Matrix - numeric features only
df.drop(columns=["train_id"], inplace=True)  # ID is non-numeric and not useful for correlation
numeric_cols = df.select_dtypes(include=np.number).columns
corr_matrix = df[numeric_cols].corr()

# Correlation with price - sorted
print("\n=== Correlation with price ===")
corr_with_price = (corr_matrix["price"]
                   .drop("price")
                   .abs()
                   .sort_values(ascending=False)
                   .round(3))

print("Feature correlation with price:")
print(corr_with_price.to_string())


=== Correlation with price ===
Feature correlation with price:
log_price            0.752
is_price_outlier     0.673
price_percentile     0.570
brand_avg_price      0.499
price_rank_in_cat    0.263
is_branded           0.138
cat_avg_price        0.135
shipping             0.100
desc_length          0.048
category_depth       0.038
name_word_count      0.036
name_length          0.033
has_description      0.028
item_condition_id    0.000
